[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/surfaces/reciprocal/reciprocal.ipynb)

# Lifts and Reciprocal Diagrams

This example computes the bar stresses and reciprocal diagram of a planar truss from a height lift of its triangulation. Assigning a vertical height to every vertex lifts a planar triangulation into a polyhedral terrain that is modelled as a piecewise-linear Airy stress function. Along each interior crease the terrain bends between adjacent triangles, and the change in slope per unit bar length gives the axial stress in that bar, positive in tension where the crease bends convex and negative in compression where it bends concave. Mapping each triangle's slope to a planar point builds a dual graph called the reciprocal diagram, whose edges cross the truss bars perpendicularly with lengths equal to the bar forces so every interior joint stays in static equilibrium while the boundary supports the external loads. Lifted onto a paraboloid, the reciprocal diagram becomes the Voronoi diagram and the bar stresses equal the cotangent weights.

Planar vertices are intrinsic `Point` antivectors and each terrain face is an extrinsic `Plane` vector whose pairing `plane & (point + up)` at unit vertical offset reads the height over `point`. Joining the opposite lifted corners of a triangle and dividing by their three-way meet `sides / (sides & corners)` forms the three corner quotients; the term ***simplex gradient*** here means each corner's linear plane contribution per unit vertex height. Pulling face planes back across interior edges via `d[1].adjugate() * planes` gives the slope jump directly as a vector perpendicular to the shared bar, and projecting out the vertical offset via `(plane ^ z) * z` isolates the horizontal slope coordinates of the reciprocal vertices.

In matrix graphic-statics notation, planar equilibrium writes the bar force densities into a symmetric stress matrix $\Omega$ with $\Omega_{ii} = -\sum_{j \neq i} \Omega_{ij}$ satisfying $\sum_j \Omega_{ij}(\mathbf{p}_i - \mathbf{p}_j) = \mathbf{0}$ at interior vertices, while the face gradients of a vertical lift $z_i$ are solved per triangle from $2\times 2$ edge-coordinate matrices and rotated by $J = \begin{pmatrix} 0 & -1 \\ 1 & 0 \end{pmatrix}$ so reciprocal edges satisfy $\mathbf{q}_L - \mathbf{q}_R = \omega_{ij}\,J(\mathbf{p}_j - \mathbf{p}_i)$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator
import numpy as np

from numga import NumpyContext, stack
from numga.algebras import VGA3D as ga
from numga.topology import Topology
from examples.surfaces.reciprocal import render, scenarios
from examples.util.animation import save_figure, show_states

mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                         # Scalar+
# Each terrain face is an extrinsic plane vector, and each vertex is an intrinsic point antivector.
Plane = ga.gatype.vector()                          # Vector+
Point = ga.gatype.antivector()                      # Antivector-
# Unit upward point offset that lifts planar points to unit height.
up = mv.z.dual()                                    # [] Point-

## Slopes, stresses, and the paraboloid lift

Each triangle of the lifted terrain spans a single plane vector holding both its horizontal slope and its height at the origin, read at any planar point by `plane & (point + up)`. Joining the two lifted corners opposite a vertex produces a plane through the origin that vanishes on both opposite corners, and dividing by its pairing with the third corner normalizes that simplex gradient so the face plane is a linear combination of the three corner heights. Because two adjacent faces meet along their shared bar, the coboundary pullback `d[1].adjugate() * planes` pairs to zero along the bar itself, leaving a pure transverse slope jump whose magnitude per unit bar length is the axial bar stress. When vertex heights lie on the paraboloid at half the squared distance from the origin, dualizing each horizontal slope vector places the reciprocal vertex at the triangle's circumcentre, and each bar stress equals half the sum of the cotangents of the two angles opposite that bar.

In [ ]:
topology, points, paraboloid = scenarios.paraboloid_grid()

In [ ]:
def face_planes(topology: Topology, points: Point, heights: Scalar) -> Plane:
    """The plane of each triangle face (`[...] Plane+[F]`) passes through its corners at `heights` (`[...] Scalar+[V]`) and pairs with `point + up` to the terrain height there."""
    lifted = points + up                                                       # Point-[V]
    corners = topology.incidence[2, 0] * lifted                                # [3] Point-[F]
    facing = topology.off[2, 0] * lifted                                       # [2] Point-[F]
    # Joining the two opposite corners gives the plane through them and the origin; dividing by its pairing with the third corner normalizes each simplex gradient to one there.
    sides = facing[0] & facing[1]                                              # [3] Plane+[F]
    gradients = sides / (sides & corners)                                      # [3] Plane+[F]
    corner_heights = topology.incidence[2, 0] * heights[..., None]             # [..., 3] Scalar+[F]
    return (gradients * corner_heights).sum(axis=-1)                           # [...] Plane+[F]


def bar_stresses(topology: Topology, points: Point, heights: Scalar) -> Scalar:
    """The axial stress on each interior bar (`[...] Scalar+[E_interior]`) equals the slope jump between its adjacent faces per unit bar length, positive in tension where the crease bends convex."""
    interior = topology.reading(topology.interior[1], 1)                       # Scalar+[E_interior <- E]
    # Pulling face planes back onto interior edges subtracts the right-hand face plane from the left-hand face plane along each oriented bar.
    jumps = interior * (topology.d[1].adjugate() * face_planes(topology, points, heights))  # [...] Plane+[E_interior]
    bars = interior * (topology.d[0] * points)                                 # Point-[E_interior]
    tails = (interior * (topology.incidence[1, 0] * points))[0]                # Point-[E_interior]
    # Project out the component along the bar to isolate the perpendicular offset toward the opposite right-hand corner.
    far_corners = topology.reading(topology.facet_opposites[:, 1], 0) * points  # Point-[E_interior]
    offsets = far_corners - tails                                              # Point-[E_interior]
    transverse = offsets - bars * ((bars | offsets) / (bars | bars))           # Point-[E_interior]
    return -(jumps & transverse) / (transverse.antinorm() * bars.antinorm())   # [...] Scalar+[E_interior]


lifted_stresses = bar_stresses(topology, points, paraboloid)                   # Scalar+[E_interior]
lifted_duals = (face_planes(topology, points, paraboloid) ^ mv.z) * mv.z       # Plane+[F]
save_figure(render.reciprocal_figure(topology, points, paraboloid, lifted_stresses, lifted_duals), "reciprocal");

On the left, the translucent polyhedral terrain rises over the paraboloid with interior creases shaded blue in tension and the outer rim drawn in grey. On the right, the dark grey Voronoi edges of the reciprocal diagram cross every stressed blue truss bar perpendicularly, shrinking toward zero length wherever four nearly cocircular vertices flatten the crease between two triangles.

## Tension and compression under breathing heights

Oscillating every vertex height harmonically around the paraboloid with an independent phase keeps every interior joint in static equilibrium while redistributing the axial forces across the bars. Wherever a vertex rises above the plane of its neighbours, the crease across the shared bar turns concave, the bar enters compression, and its reciprocal edge reverses direction so the dual cells fold across one another while remaining perpendicular to the primal bars.

In graphic-statics terminology, the planar truss is called the form diagram and the reciprocal diagram is called the force diagram, where the closed polygon of reciprocal edges around each face of the dual complex expresses vector force balance $\sum_{j} \mathbf{f}_{ij} = \mathbf{0}$ at the corresponding interior truss joint.

In [ ]:
phases = scenarios.vertex_phases(topology)                                     # [V]
amplitude = scenarios.amplitude
n_frames = scenarios.n_frames
frame_duration_ms = scenarios.frame_duration_ms

In [ ]:
def breathing_states(
    topology: Topology,
    points: Point,
    base_heights: Scalar,
    phases: np.ndarray,
    amplitude: float,
    n_frames: int,
) -> Iterator[tuple[Scalar, Scalar, Plane]]:
    """Yields `(heights, stresses, dual_vertices)` (`(Scalar+[V], Scalar+[E_interior], Plane+[F])`) across `n_frames` frames of one harmonic oscillation around `base_heights`."""
    for step in range(n_frames):
        angle = 2 * np.pi * step / n_frames
        heights = base_heights + mv.scalar((amplitude * np.sin(angle + phases))[:, None]).field()  # Scalar+[V]
        stresses = bar_stresses(topology, points, heights)                                         # Scalar+[E_interior]
        dual_vertices = (face_planes(topology, points, heights) ^ mv.z) * mv.z                     # Plane+[F]
        yield heights, stresses, dual_vertices


recorded = show_states(
    breathing_states(topology, points, paraboloid, phases, amplitude, n_frames),
    render.breath_animation,
    topology,
    points,
    name="reciprocal_breath",
    duration_ms=frame_duration_ms,
    n_frames=n_frames,
);

Blue bars and creases carry tension across convex ridges while red bars mark concave creases in compression, where the dark grey reciprocal edges flip orientation and overlap across neighbouring dual polygons.

In [ ]:
breaths = stack([state[0] for state in recorded])                              # [n_frames] Scalar+[V]
breath_stresses = stack([state[1] for state in recorded])                      # [n_frames] Scalar+[E_interior]
breath_duals = stack([state[2] for state in recorded])                         # [n_frames] Plane+[F]
compressed = (breath_stresses < 0).sum(axis=-1)                                # [n_frames]
print(f"bars in compression over the breath: {compressed.min()} to {compressed.max()} of {len(topology.interior[1])}")

In [ ]:
# checks
def edge_folds(topology: Topology, points: Point, heights: Scalar) -> Scalar:
    """The vertical fold across each interior edge (`[...] Scalar+[E_interior]`) measures how far the opposite right-hand corner rises above the left-hand face plane."""
    interior = topology.reading(topology.interior[1], 1)                       # Scalar+[E_interior <- E]
    # Because the right-hand face plane passes through that opposite corner at its lifted height, pairing the slope jump at unit height reads the vertical clearance directly.
    jumps = interior * (topology.d[1].adjugate() * face_planes(topology, points, heights))  # [...] Plane+[E_interior]
    lifted_corners = topology.reading(topology.facet_opposites[:, 1], 0) * (points + up)    # Point-[E_interior]
    return -(jumps & lifted_corners)                                           # [...] Scalar+[E_interior]


folds = edge_folds(topology, points, breaths)                                  # [n_frames] Scalar+[E_interior]
# Every bar carries tension exactly where the terrain crease bends convex, while concave creases enter compression.
assert np.array_equal(breath_stresses > 0, folds > 0)
assert (breath_stresses < 0).any()
# Every reciprocal edge is orthogonal to its primal bar because adjacent face planes agree along their shared edge.
interior = topology.reading(topology.interior[1], 1)                           # Scalar+[E_interior <- E]
bars = interior * (topology.d[0] * points)                                     # Point-[E_interior]
assert (((interior * (topology.d[1].adjugate() * breath_duals)) & bars).abs() < 1e-10).all()
# Summing axial bar forces onto vertices leaves every interior truss joint in static equilibrium.
pulls = bars * breath_stresses                                                 # [n_frames] Point-[E_interior]
bar_ends = interior * topology.d[0]                                            # Scalar+[E_interior <- V]
net_forces = -(bar_ends.adjugate() * pulls)                                    # [n_frames] Point-[V]
assert ((topology.reading(topology.interior[0], 0) * net_forces).antinorm() < 1e-10).all()
# On the paraboloid lift, bar stresses equal the cotangent weights and reciprocal vertices lie at the triangle circumcentres.
directed_sides = topology.boundary[2, 1] * (topology.d[0] * points)            # [3] Point-[F]
first_sides, second_sides = topology.corners[2, 1]                             # [3], [3]
after, before = directed_sides[first_sides], directed_sides[second_sides]      # [3] Point-[F], [3] Point-[F]
cotangents = (after | before) / (after & before).antinorm()                    # [3] Scalar+[F]
cotangent_weights = (topology.incidence[2, 1].adjugate() * (cotangents / 2)).sum(axis=0)  # Scalar+[E]
assert ((lifted_stresses - interior * cotangent_weights).abs() < 1e-10).all()
corners = topology.incidence[2, 0] * points                                    # [3] Point-[F]
squared_distances = (lifted_duals.dual() - corners).scalar_norm_squared()      # [3] Scalar+[F]
assert ((squared_distances[0] - squared_distances[1]).abs() < 1e-10).all()
assert ((squared_distances[0] - squared_distances[2]).abs() < 1e-10).all()